In [19]:
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
    KFold,
    RandomizedSearchCV,
    StratifiedKFold
)
from sklearn.compose import (
    ColumnTransformer,
    make_column_selector
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder, 
    OrdinalEncoder,
    StandardScaler, 
    FunctionTransformer
)

In [20]:
#load data
loans_data = pd.read_csv("../data/loan_data_raw.csv")

In [21]:
loans_data.head()


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0.0,RENT,35000.0,NaN,16.02,0.49,3.0,561.0,No,1
1,21.0,female,High School,12282.0,0.0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504.0,Yes,0
2,25.0,female,High School,12438.0,3.0,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635.0,No,1
3,23.0,female,Bachelor,79753.0,0.0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675.0,No,1
4,24.0,male,Master,66135.0,1.0,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586.0,No,1


In [22]:
loans_data.describe()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,45000.000000,4.356000e+04,43740.000000,45000.000000,43470.000000,45000.000000,44191.000000,43560.000000,45000.000000
mean,27.764178,8.032358e+04,5.413900,9583.157556,11.004356,0.139725,5.862257,632.619284,0.222222
std,6.045108,8.121192e+04,6.065794,6314.886691,2.978830,0.087212,3.875905,50.495707,0.415744
min,20.000000,8.000000e+03,0.000000,500.000000,5.420000,0.000000,2.000000,390.000000,0.000000
25%,24.000000,4.710850e+04,1.000000,5000.000000,8.590000,0.070000,3.000000,601.000000,0.000000
50%,26.000000,6.704000e+04,4.000000,8000.000000,11.010000,0.120000,4.000000,640.000000,0.000000
75%,30.000000,9.575725e+04,8.000000,12237.250000,12.990000,0.190000,8.000000,670.000000,0.000000
max,144.000000,7.200766e+06,125.000000,35000.000000,20.000000,0.660000,30.000000,850.000000,1.000000


In [23]:
#unusually high ages in the dataset, let's check them out

loans_data[loans_data["person_age"] > 80]["person_age"].value_counts().sort_index()

person_age
84.0     1
94.0     1
109.0    1
116.0    1
123.0    2
144.0    3
Name: count, dtype: int64

In [24]:
loans_data = loans_data[loans_data["person_age"] <= 85].copy()

In [25]:
# split data

loan_train_set, loan_test_set = train_test_split(
    loans_data,
    test_size=0.2,
    stratify=loans_data["loan_status"], 
    random_state=42
)

# drop person_gender (fairness concerns, shown low feature importantce)
loan_train_set = loan_train_set.drop(columns=["person_gender"])
loan_test_set = loan_test_set.drop(columns=["person_gender"])

# split labels and features for train and test sets
x_train = loan_train_set.drop("loan_status", axis=1)
y_train = loan_train_set["loan_status"].copy()

x_test = loan_test_set.drop("loan_status", axis=1)
y_test = loan_test_set["loan_status"].copy()

In [26]:
# TRANSFORMATION PIPELINE

# func for ratio of 2 columns
def column_ratio(X):
    result = X[:, [0]] / X[:, [1]]
    result[~np.isfinite(result)] = np.nan
    return result

# func for naming ratio column
def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

# pipeline for ratio of 2 columns
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        SimpleImputer(strategy="median"),
    )

# pipeline for numerical columns
default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
)

# categorical pipeline (excludes person_gender and person_education)
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)

# ordinal pipeline for education (natural order)
education_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OrdinalEncoder(categories=[["High School", "Associate", "Bachelor", "Master", "Doctorate"]]),
)

# explicit categorical columns (no gender, no education)
cat_cols = ["person_home_ownership", "loan_intent", "previous_loan_defaults_on_file"]

# full preprocessing pipeline
preprocessing = ColumnTransformer(
    [
        # ordinal education instead of one-hot
        ("education", education_pipeline, ["person_education"]),
        # categoricals (no gender, no education)
        ("cat", cat_pipeline, cat_cols),
    ],
    remainder=default_num_pipeline,
)

print(preprocessing)

ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                             SimpleImputer(strategy='median'))]),
                  transformers=[('education',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(categories=[['High '
                                                                              'School',
                                                                              'Associate',
                                                                              'Bachelor',
                                                                              'Master',
                                                                              'Doctorate']]))]),
                                 ['person_e

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# full pipeline: preprocessing -> random forest
full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestClassifier(random_state=42, n_jobs=-1))
])

# hyperparameter search space
param_distributions = {
    "random_forest__n_estimators": randint(100, 500),
    "random_forest__max_depth": randint(3, 20),
    "random_forest__min_samples_split": randint(2, 10),
    "random_forest__min_samples_leaf": randint(1, 5),
    "random_forest__max_features": randint(2, 10),
}

# cv variable
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# multiple scoring metrics for evaluation
scoring = {"f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision",
           "precision": "precision", "recall": "recall"}

# randomised search with 5-fold CV
rnd_search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    cv=cv,
    scoring=scoring,
    refit= "f1",  # refit the best model based on F1 score
    random_state=42,
    n_jobs=-1
)

# fit only on training data
rnd_search.fit(x_train, y_train)

print("Best params:", rnd_search.best_params_)
print("Best CV F1:", f"{rnd_search.best_score_:.4f}")

/Users/kirillshatilov/loanproject/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/kirillshatilov/loanproject/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/kirillshatilov/loanproject/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/kirillshatilov/

Best params: {'random_forest__max_depth': 18, 'random_forest__max_features': 8, 'random_forest__min_samples_leaf': 2, 'random_forest__min_samples_split': 7, 'random_forest__n_estimators': 274}
Best CV F1: 0.8167


In [28]:
# check feature importances

rf_model = rnd_search.best_estimator_
feature_importances = rf_model["random_forest"].feature_importances_
feature_names = rf_model["preprocessing"].get_feature_names_out()

feat_imp_asc = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values(by="importance", ascending=True)

feat_imp_desc = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values(by="importance", ascending=False)

print(feat_imp_desc.to_string(index=False))


                                feature  importance
         remainder__loan_percent_income    0.172624
cat__previous_loan_defaults_on_file_Yes    0.168499
               remainder__loan_int_rate    0.158450
 cat__previous_loan_defaults_on_file_No    0.142807
               remainder__person_income    0.110677
        cat__person_home_ownership_RENT    0.044405
                remainder__credit_score    0.041835
                   remainder__loan_amnt    0.037540
                  remainder__person_age    0.017697
    cat__person_home_ownership_MORTGAGE    0.015634
              remainder__person_emp_exp    0.015505
  remainder__cb_person_cred_hist_length    0.014259
         cat__person_home_ownership_OWN    0.009912
       cat__loan_intent_HOMEIMPROVEMENT    0.009477
     cat__loan_intent_DEBTCONSOLIDATION    0.008781
               cat__loan_intent_MEDICAL    0.008655
            education__person_education    0.008466
               cat__loan_intent_VENTURE    0.006821
            